# Fine-tuning sur Colab — captures réelles (pipeline retrain.py)

Reproduit **exactement** le pipeline local, sur GPU Colab (T4) : split par disposition
(test compost jamais appris) → éval **avant** → fine-tuning → éval **après**, via
`scripts/retrain.py` — aucune logique dupliquée dans ce notebook.

**Prérequis (une seule fois)** :
- secret `GITHUB_TOKEN` dans Colab (icône clé à gauche) ;
- le zip du snapshot sur Drive : `MyDrive/compost/dataset_captures_v002.zip` ;
- GPU activé : Exécution → Modifier le type d'exécution → T4.

Durée indicative : **~1 h** (RT-DETR batch 4). Si Colab coupe : relancer la cellule 5
(le split est déterministe, le run repart de zéro).


In [ ]:
# 1. Clone du repo
BRANCH = 'centralisation'   # branche de travail ; mettre 'main' après fusion
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
%cd /content
!rm -rf /content/repo
!git clone --depth 1 --branch {BRANCH} https://{token}@github.com/TSResearch-hub/Compost_Waste_Yolo.git /content/repo
%cd /content/repo/compost-yolo


In [ ]:
# 2. Installation des dépendances
!pip install -q -e .


In [ ]:
# 3. Paramètres + montage Drive
# Pré-entraîné canonique (même rôle que models/ en local) — v2 RT-DETR, 100 epochs :
PRETRAIN_PATH = '/content/drive/MyDrive/compost/backups/pretrain_16-07_03h44/weights/best.pt'
# Zip du snapshot de captures (copie de data/captures/vNNN du PC, voir README) :
SNAPSHOT_ZIP  = '/content/drive/MyDrive/compost/dataset_captures_v002.zip'
EPOCHS = 30   # ~25-30 suffisent sur ~440 images (au-delà : sur-apprentissage constaté)
BATCH  = 4    # RT-DETR s'entraîne à 1280 px (mosaïque) : 4 tient sur T4 ; YOLO : 16

import os, shutil
from google.colab import drive
if os.path.isdir('/content/drive') and not os.path.ismount('/content/drive'):
    shutil.rmtree('/content/drive', ignore_errors=True)
drive.mount('/content/drive')
assert os.path.exists(PRETRAIN_PATH), f'Pré-entraîné introuvable : {PRETRAIN_PATH}'
assert os.path.exists(SNAPSHOT_ZIP),  f'Zip du snapshot introuvable : {SNAPSHOT_ZIP}'


In [ ]:
# 4. Décompression du snapshot + contrôle (mêmes comptes que le snapshot local)
!rm -rf /content/captures_snapshot && mkdir -p /content/captures_snapshot
!unzip -q {SNAPSHOT_ZIP} -d /content/captures_snapshot
from pathlib import Path
snap = Path('/content/captures_snapshot')
if not (snap / 'images').is_dir():   # zip fait depuis le dossier parent du snapshot
    snap = next(d for d in snap.iterdir() if (d / 'images').is_dir())
n_img = sum(1 for f in (snap / 'images').iterdir()
            if f.suffix.lower() in ('.jpg', '.jpeg', '.png'))
n_lbl = len(list((snap / 'labels').glob('*.txt')))
print(f'{n_img} images, {n_lbl} labels — attendu : 440 / 279 (snapshot v002_09-07)')
assert (snap / 'groups.csv').exists(), 'groups.csv manquant : split par disposition impossible'
assert (n_img, n_lbl) == (440, 279), 'Comptes inattendus — mauvais zip ? (adapter si nouveau snapshot)'
SNAP = str(snap)


In [ ]:
# 5. Pipeline complet — identique au local (split -> éval avant -> fine-tuning -> éval après).
#    La comparaison avant/après s'affiche à la fin de la sortie.
!python scripts/retrain.py --pretrain {PRETRAIN_PATH} --captures {SNAP} \
    --epochs {EPOCHS} --batch {BATCH} --runs-dir /content/runs


In [ ]:
# 6. Figures des évaluations (avant = eval_pretrain_*, après = eval_finetune_*)
from pathlib import Path
from IPython.display import Image, display
for d in sorted(Path('/content/runs').glob('eval_*')):
    print('\n===', d.name)
    for png in ('per_class_metrics.png', 'confusion_matrices.png'):
        if (d / png).exists():
            display(Image(str(d / png), width=700))


In [ ]:
# 7. Sauvegarde des runs sur Drive + téléchargement direct du best.pt
!mkdir -p /content/drive/MyDrive/compost/runs
!cp -r /content/runs/* /content/drive/MyDrive/compost/runs/
from pathlib import Path
best = max(Path('/content/runs').glob('finetune_*/weights/best.pt'),
           key=lambda p: p.stat().st_mtime)
print('best.pt :', best)
from google.colab import files
files.download(str(best))   # arrive dans les téléchargements du navigateur (~252 Mo)


## Déployer sur le PC (pré-annotation dans l'interface)

Placer le `best.pt` téléchargé comme modèle des interfaces :

```bash
cp /mnt/c/Users/ikche/Downloads/best.pt ~/stage/Compost_Waste_Yolo/weights/best.pt
```

puis relancer l'interface d'annotation. Les runs complets (courbes, évals) sont aussi
sur Drive dans `compost/runs/` — à rapatrier dans `runs/` local pour l'onglet Résultats.
